# Smart IVC cage: reproduce the behaviour classifier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aghasalim/smart-ivc-cage-platform/blob/main/ai/training/behaviour_classifier_colab.ipynb)

This notebook runs the whole pipeline in `ai/training/` on a free Colab runtime: generate the synthetic set, train the production model, compare three candidates, and draw the confusion matrix. It takes about two minutes.

Read the README's section *The number I am not going to advertise* before you quote anything from here. The last cell shows why: a plain decision tree gets the same score, because the data was written by rules.

In [ ]:
!git clone --depth 1 https://github.com/aghasalim/smart-ivc-cage-platform.git
%cd smart-ivc-cage-platform/ai
!pip install -q -r requirements.txt

## 1. Generate the dataset

Seeded, so you get the same 20 000 rows that are committed in `ai/data/synthetic.csv`. See `ai/data/DATASET.md` for what each column is.

In [ ]:
!python training/generate_dataset.py

import pandas as pd
df = pd.read_csv('data/synthetic.csv')
print(df.shape)
df['label'].value_counts()

## 2. Train the production model

`HistGradientBoostingClassifier`, stratified 85/15/15 split, seed 42. Writes `models/behaviour.pkl` and `reports/metrics.json`.

In [ ]:
!python training/train.py
!python training/compare_models.py

import json
m = json.load(open('reports/metrics.json'))
print(f"val  macro-F1 {m['val_macro_f1']:.4f}")
print(f"test macro-F1 {m['test_macro_f1']:.4f}")
print(open('reports/model-comparison.md').read())

## 3. Confusion matrix

The only confusions are resting ↔ sleeping, which share a `movement_cm` boundary in the generator.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cm = np.array(m['test_confusion_matrix'])
classes = m['classes']
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(classes)), classes, rotation=45, ha='right')
ax.set_yticks(range(len(classes)), classes)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout(); plt.show()

## 4. Why the score is not impressive

An unpruned decision tree, no tuning, on the same split. If it lands within a few thousandths of the boosted model, the classes were separable by construction, and the model is just reading the generator's rules back.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

y = df['label'].values
X = df.drop(columns=['label']).values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
tree = DecisionTreeClassifier(random_state=42).fit(Xtr, ytr)
print(f"decision tree macro-F1: {f1_score(yte, tree.predict(Xte), average='macro'):.4f}")
print(f"boosted model macro-F1: {m['test_macro_f1']:.4f}")

# the single feature that does most of the work
df.groupby('label')['movement_cm'].agg(['min', 'median', 'max']).sort_values('median')

## 5. Take the model with you

The backend loads `backend/models/behaviour.pkl` on boot and falls back to rules if it's missing. Download it and drop it there, then rebuild the backend image.

In [ ]:
from google.colab import files
files.download('models/behaviour.pkl')

## What would make this real

Labelled recordings from the actual cage. The feature vector the backend sends to the model is the same one the generator writes (`grid_0..15`, `movement_cm`, `weight_g`, `gate_open`, `hour`, `minute`), so a few hours of hand-labelled minutes exported from the dashboard (`/api/v1/cages/{id}/export.csv`) could replace `synthetic.csv` with no code change.